# COLAB SETUP

In [ ]:
!git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git


In [ ]:
# Run this cell first if you are executing the notebook in Google Colab.
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")

    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas seaborn')

    from google.colab import drive
    drive.mount('/content/drive')

    import os, sys

    possible_paths = [
        '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Generalisation_BB_WB',
        os.path.join(os.getcwd(), 'Black_vs_White_Lie_Detection', 'Generalisation_BB_WB'),
        os.getcwd()
    ]

    base_proj_path = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, 'generalization_utils.py')):
            base_proj_path = path
            break

    if base_proj_path:
        os.chdir(base_proj_path)
        print(f"Changed working directory to: {base_proj_path}")
    else:
        print("Could not find project directory. Check your file structure.")

    current_dir   = os.getcwd()
    aligned_dir   = os.path.join(current_dir, '..', 'Aligned_Comparison_BB_WB')
    parent_dir    = os.path.dirname(current_dir)
    project_root  = os.path.dirname(parent_dir)

    for p in [current_dir, aligned_dir, parent_dir, project_root]:
        p = os.path.normpath(p)
        if p not in sys.path:
            sys.path.insert(0, p)

    print(f"sys.path updated.")

    try:
        from huggingface_hub import login
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            login(token=hf_token)
            print("Authenticated with Hugging Face.")
        else:
            login()
    except Exception as e:
        print(f"HF login warning: {e}")

except ImportError:
    IN_COLAB = False
    print("Running locally. Setup skipped.")


# Imports and Settings

In [ ]:
import torch, numpy as np, pandas as pd, pickle, os, time, gc, re
from tqdm.auto import tqdm

from load_data import load_dataset_aligned
from wb_activations import load_model, get_activations_for_dataset
from generalization_utils import (
    evaluate_bb_on_target,
    evaluate_wb_on_target,
    compute_recovered_metrics,
    plot_recovery_heatmap,
)

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge",
    "open_book_qa", "got_cities", "got_sp_en_trans", "got_larger_than", "imdb",
    "amazon_polarity", "ag_news", "dbpedia_14", "rte", "copa", "boolq",
]

# race/copa/boolq use corrected prompts — classifiers and features have _new suffix
FIXED_MAP = {"race": "race_new", "copa": "copa_new", "boolq": "boolq_new"}

E3_DIR      = os.path.join("..", "Aligned_Comparison_BB_WB")
L2_FEAT_DIR = os.path.join(E3_DIR, "bb_elicitation_vectors_lvl_2")
E2_CACHE    = os.path.join("..", "Data_Efficiency_BB_WB", "outputs")
L2_CLF_DIR  = os.path.join(E3_DIR, "classifiers", "level2")
L3_CLF_DIR  = os.path.join(E3_DIR, "classifiers", "level3")
COMPACT_CSV = os.path.join(E3_DIR, "outputs", "compact_comparison_results_final.csv")

METHOD_NAMES = {
    "lr":    "Logistic Regression",
    "dim":   "Difference in Means",
    "lat":   "Linear Artificial Tomography",
    "ccs":   "Contrast-Consistent Search",
    "lda":   "Linear Discriminant Analysis",
    "pca":   "Principal Component Analysis",
    "pca-g": "Grouped PCA",
    "lr-g":  "Grouped Logistic Regression",
}
METHOD_KEYS = {v: k for k, v in METHOD_NAMES.items()}

def parse_best_l3(s):
    m = re.match(r"[\d.]+ \((.+), L(\d+)\)", str(s))
    if m:
        return m.group(1), int(m.group(2))
    return None, None

compact_df = pd.read_csv(COMPACT_CSV)
BEST_WB = {}
for _, row in compact_df.iterrows():
    ds = row["Dataset"]
    method_name, layer = parse_best_l3(row["Best L3 WB AUC"])
    if method_name and layer:
        BEST_WB[ds] = {
            "method_key": METHOD_KEYS.get(method_name),
            "layer":      layer,
        }

LAYERS_TO_EXTRACT = sorted({info['layer'] for info in BEST_WB.values()})
print(f"Device: {DEVICE}")
print(f"Best WB info loaded for {len(BEST_WB)} datasets")
print(f"Unique WB layers needed: {LAYERS_TO_EXTRACT}")


# 1. Load Datasets (all 15)

In [ ]:
dataset_dict = {}
print(f"Loading {len(DATASETS)} datasets...")
start = time.time()

for ds_name in tqdm(DATASETS, desc='Step 1 -- Loading datasets'):
    ds_train = load_dataset_aligned(ds_name, split='train',      n_samples=5000)
    ds_val   = load_dataset_aligned(ds_name, split='validation', n_samples=1000)
    df_train = pd.DataFrame(ds_train)
    df_val   = pd.DataFrame(ds_val)
    df = pd.concat([df_train, df_val], ignore_index=True)
    if len(df_train) > 0 and len(df_val) > 0:
        dataset_dict[ds_name] = df
        print(f'  {ds_name}: {len(df):,} samples  (train={len(df_train):,}, val={len(df_val):,})')
    else:
        print(f'  Skipped {ds_name}: missing data.')

print(f'\n Step 1 Complete: {len(dataset_dict)}/{len(DATASETS)} datasets in {time.time()-start:.1f}s')


# 2. Load Model

In [ ]:
print('Step 2: Loading model...')
start = time.time()
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)
print(f' Step 2 Complete: Model loaded in {time.time()-start:.1f}s')


# 3. Extract WB Activations (RAM only — no disk cache)

Only the layers needed by the best WB probe per dataset are extracted.

In [ ]:
wb_act_dict = {}

print(f'Step 3: Extracting WB activations (layers={LAYERS_TO_EXTRACT})...')
print('  Stored in RAM only -- no disk write.')
start = time.time()

for ds_name in tqdm(DATASETS, desc='Step 3 -- WB Activations'):
    if ds_name not in dataset_dict:
        continue
    df       = dataset_dict[ds_name]
    df_train = df[df['split'] == 'train'].reset_index(drop=True)
    df_val   = df[df['split'] == 'validation'].reset_index(drop=True)

    bs = 28 if ds_name in ('imdb', 'race') else 64

    raw_train = get_activations_for_dataset(df_train, model, tokenizer, device=DEVICE,
                                            batch_size=bs, show_progress=True)
    raw_val   = get_activations_for_dataset(df_val,   model, tokenizer, device=DEVICE,
                                            batch_size=bs, show_progress=True)

    def to_layer_arrays(raw, layers):
        return {
            'X': {l: np.array([item['activations'][l] for item in raw], dtype=np.float16)
                  for l in layers},
            'y': np.array([item['label'] for item in raw]),
        }

    wb_act_dict[ds_name] = {
        'train':      to_layer_arrays(raw_train, LAYERS_TO_EXTRACT),
        'validation': to_layer_arrays(raw_val,   LAYERS_TO_EXTRACT),
    }
    print(f'  {ds_name}: {len(raw_train)} train, {len(raw_val)} val activations')
    gc.collect()

print(f'\n Step 3 Complete: WB activations in RAM for {len(wb_act_dict)} datasets in {time.time()-start:.1f}s')


# 4. Load BB Level 2 Features

Priority: `bb_elicitation_vectors_lvl_2/` (newest, includes `_new` for race/copa/boolq) then `Data_Efficiency_BB_WB/outputs/`.

In [ ]:
l2_feat_dict = {}

print('Step 4: Loading BB Level 2 features...')
start = time.time()

for ds_name in tqdm(DATASETS, desc='Step 4 -- L2 Features'):
    clf_name = FIXED_MAP.get(ds_name, ds_name)  # race_new / copa_new / boolq_new

    p1 = os.path.join(L2_FEAT_DIR, f'{clf_name}_bb_features_lvl2.pkl')  # _new local
    p2 = os.path.join(L2_FEAT_DIR, f'{ds_name}_bb_features_lvl2.pkl')   # plain local
    p3 = os.path.join(E2_CACHE,    f'{ds_name}_bb_features_efficiency.pkl')  # Data_Efficiency

    if os.path.exists(p1):
        with open(p1, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train']
        X_val,   y_val   = data['validation']
        src = 'local (_new)'
    elif os.path.exists(p2):
        with open(p2, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train']
        X_val,   y_val   = data['validation']
        src = 'local'
    elif os.path.exists(p3):
        with open(p3, 'rb') as f: data = pickle.load(f)
        X_train, y_train = data['train_full']
        X_val,   y_val   = data['validation']
        src = 'Data_Efficiency'
    else:
        print(f'  No L2 features found for {ds_name} -- skipping.')
        continue

    l2_feat_dict[ds_name] = {
        'X_train': X_train, 'y_train': y_train,
        'X_val':   X_val,   'y_val':   y_val,
    }
    print(f'  {ds_name}: {src}  (train={len(X_train)}, val={len(X_val)})')

print(f'\n Step 4 Complete: L2 features for {len(l2_feat_dict)}/{len(DATASETS)} datasets in {time.time()-start:.1f}s')


# 5. Cross-Dataset Evaluation

**L2 (BB):** apply each source classifier on the validation set of all 14 other datasets.

**L3 (WB):** for each source, use the best method + best layer (from `compact_comparison_results_final.csv`) and evaluate on all 14 other datasets.

In [ ]:
# Load E3 in-distribution baselines
l2_csv_path = os.path.join(E3_DIR, 'outputs', 'level2_results.csv')
if not os.path.exists(l2_csv_path):
    l2_csv_path = os.path.join(E3_DIR, 'level2_results.csv')  # fallback
df_l2_indist = pd.read_csv(l2_csv_path)

l3_csv_path = os.path.join(E3_DIR, 'outputs', 'level3_results.csv')
df_l3_indist = pd.read_csv(l3_csv_path)

def get_l2_indist(ds_name):
    clf_name = FIXED_MAP.get(ds_name, ds_name)
    row = df_l2_indist[df_l2_indist['Dataset'] == clf_name]
    return {k: row[k].values[0] for k in ['AUC', 'MAP', 'BRP_90']} if not row.empty else None

def get_l3_indist(ds_name, method_key):
    clf_name = FIXED_MAP.get(ds_name, ds_name)
    rows = df_l3_indist[
        (df_l3_indist['Dataset'] == clf_name) &
        (df_l3_indist['Method']  == METHOD_NAMES[method_key])
    ]
    return {k: rows[k].values[0] for k in ['AUC', 'MAP', 'BRP_90']} if not rows.empty else None

print('In-distribution baselines loaded.')
print(f'  L2 CSV: {l2_csv_path}  ({len(df_l2_indist)} rows)')
print(f'  L3 CSV: {l3_csv_path}  ({len(df_l3_indist)} rows)')


In [ ]:
os.makedirs('outputs', exist_ok=True)

print('Step 5a: BB L2 Cross-Dataset Evaluation...')
start = time.time()
l2_results = []
available_l2 = [ds for ds in DATASETS if ds in l2_feat_dict]

for source_ds in tqdm(available_l2, desc='L2 -- source'):
    clf_name = FIXED_MAP.get(source_ds, source_ds)
    clf_path = os.path.join(L2_CLF_DIR, f'{clf_name}_l2_classifier.pkl')
    if not os.path.exists(clf_path):
        print(f'  No L2 classifier for {source_ds} -- skipping.')
        continue
    with open(clf_path, 'rb') as f:
        bundle = pickle.load(f)

    in_dist = get_l2_indist(source_ds)
    if in_dist is None:
        print(f'  No in-dist baseline for {source_ds} -- skipping.')
        continue

    for target_ds in available_l2:
        y_t = l2_feat_dict[target_ds]['y_val']
        X_t = l2_feat_dict[target_ds]['X_val']
        out_dist = in_dist if target_ds == source_ds else evaluate_bb_on_target(bundle, X_t, y_t)
        rec = compute_recovered_metrics(in_dist, out_dist)
        l2_results.append({
            'Source': source_ds, 'Target': target_ds,
            'InDist_AUC':    in_dist['AUC'],   'InDist_MAP':    in_dist['MAP'],   'InDist_BRP_90':    in_dist['BRP_90'],
            'OutDist_AUC':   out_dist['AUC'],  'OutDist_MAP':   out_dist['MAP'],  'OutDist_BRP_90':   out_dist['BRP_90'],
            'Recovered_AUC': rec['Recovered_AUC'],
            'Recovered_MAP': rec['Recovered_MAP'],
            'Recovered_BRP_90': rec['Recovered_BRP_90'],
        })

df_l2_results = pd.DataFrame(l2_results)
df_l2_results.to_csv('outputs/generalisation_l2_results.csv', index=False)
print(f'\n Step 5a Complete: {len(df_l2_results)} rows in {time.time()-start:.1f}s')
print('  -> outputs/generalisation_l2_results.csv')


In [ ]:
print('Step 5b: WB L3 Cross-Dataset Evaluation (best method per source)...')
start = time.time()
l3_results = []
available_l3 = [ds for ds in DATASETS if ds in wb_act_dict and ds in BEST_WB]

for source_ds in tqdm(available_l3, desc='L3 -- source'):
    info       = BEST_WB[source_ds]
    method_key = info['method_key']
    best_layer = info['layer']
    clf_name   = FIXED_MAP.get(source_ds, source_ds)
    clf_path   = os.path.join(L3_CLF_DIR, f'{clf_name}_{method_key}_l3_classifier.pkl')

    if not os.path.exists(clf_path):
        print(f'  No L3 classifier for {source_ds}/{method_key} -- skipping.')
        continue
    with open(clf_path, 'rb') as f:
        probe = pickle.load(f)

    in_dist = get_l3_indist(source_ds, method_key)
    if in_dist is None:
        print(f'  No in-dist L3 baseline for {source_ds}/{method_key} -- skipping.')
        continue

    X_src_train = wb_act_dict[source_ds]['train']['X'][best_layer].astype(np.float32)
    y_src_train = wb_act_dict[source_ds]['train']['y']

    for target_ds in available_l3:
        X_t = wb_act_dict[target_ds]['validation']['X'][best_layer].astype(np.float32)
        y_t = wb_act_dict[target_ds]['validation']['y']
        out_dist = in_dist if target_ds == source_ds else evaluate_wb_on_target(
            probe, method_key, X_src_train, y_src_train, X_t, y_t
        )
        rec = compute_recovered_metrics(in_dist, out_dist)
        l3_results.append({
            'Source': source_ds, 'Target': target_ds,
            'Method': METHOD_NAMES[method_key], 'Layer': best_layer,
            'InDist_AUC':    in_dist['AUC'],   'InDist_MAP':    in_dist['MAP'],   'InDist_BRP_90':    in_dist['BRP_90'],
            'OutDist_AUC':   out_dist['AUC'],  'OutDist_MAP':   out_dist['MAP'],  'OutDist_BRP_90':   out_dist['BRP_90'],
            'Recovered_AUC': rec['Recovered_AUC'],
            'Recovered_MAP': rec['Recovered_MAP'],
            'Recovered_BRP_90': rec['Recovered_BRP_90'],
        })

df_l3_results = pd.DataFrame(l3_results)
df_l3_results.to_csv('outputs/generalisation_l3_results.csv', index=False)
print(f'\n Step 5b Complete: {len(df_l3_results)} rows in {time.time()-start:.1f}s')
print('  -> outputs/generalisation_l3_results.csv')


# 6. Recovery Heatmaps

Heatmap 1 — BB L2 recovered AUC.
Heatmap 2 — WB L3 recovered AUC (best method per source dataset).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print('Step 6: Plotting heatmaps...')
os.makedirs('outputs', exist_ok=True)

df_l2_res = pd.read_csv('outputs/generalisation_l2_results.csv')
df_l3_res = pd.read_csv('outputs/generalisation_l3_results.csv')

ds_order_l2 = [ds for ds in DATASETS if ds in df_l2_res['Source'].unique()]
ds_order_l3 = [ds for ds in DATASETS if ds in df_l3_res['Source'].unique()]

# Heatmap 1: BB L2
pivot_l2 = (
    df_l2_res
    .groupby(['Source', 'Target'])['Recovered_AUC']
    .mean()
    .unstack('Target')
    .reindex(index=ds_order_l2, columns=ds_order_l2)
)
plot_recovery_heatmap(
    pivot_l2,
    'Recovered AUC (BB L2)',
    'BB L2 Cross-Dataset Generalisation -- Recovered AUC',
    'outputs/generalisation_l2_recovered_auc.png',
)

# Heatmap 2: WB L3 best method
pivot_l3 = (
    df_l3_res
    .groupby(['Source', 'Target'])['Recovered_AUC']
    .mean()
    .unstack('Target')
    .reindex(index=ds_order_l3, columns=ds_order_l3)
)
plot_recovery_heatmap(
    pivot_l3,
    'Recovered AUC (WB L3 -- best method)',
    'WB L3 Cross-Dataset Generalisation -- Recovered AUC (best method per source)',
    'outputs/generalisation_l3_recovered_auc.png',
)

print('\n Step 6 Complete: 2 heatmaps saved to outputs/')


## RQ2 Direct Answer — BB L2 vs WB L3 Generalisation

Off-diagonal recovered AUC only (diagonal is always 1.0 by construction).

In [ ]:
# Off-diagonal only: source != target
df_l2_off = df_l2_res[df_l2_res['Source'] != df_l2_res['Target']].copy()
df_l3_off = df_l3_res[df_l3_res['Source'] != df_l3_res['Target']].copy()

# Mean recovered AUC per source dataset
l2_per_source = df_l2_off.groupby('Source')['Recovered_AUC'].mean().rename('BB L2')
l3_per_source = df_l3_off.groupby('Source')['Recovered_AUC'].mean().rename('WB L3 (best)')
comparison_df = pd.concat([l2_per_source, l3_per_source], axis=1)
comparison_df['WB - BB'] = comparison_df['WB L3 (best)'] - comparison_df['BB L2']
comparison_df = comparison_df.sort_values('WB - BB', ascending=False)

print('=== RQ2: Mean recovered AUC per source (off-diagonal) ===')
print(comparison_df.round(3).to_string())
print()
mean_bb = df_l2_off['Recovered_AUC'].mean()
mean_wb = df_l3_off['Recovered_AUC'].mean()
print(f'Global mean  BB L2       : {mean_bb:.3f}')
print(f'Global mean  WB L3 (best): {mean_wb:.3f}')
winner = 'WB L3' if mean_wb > mean_bb else 'BB L2'
print(f'=> {winner} generalises better on average (delta = {abs(mean_wb - mean_bb):.3f} AUC)')
comparison_df.to_csv('outputs/rq2_bb_vs_wb_generalisation.csv')

# Bar chart: BB L2 vs WB L3 (best) per source dataset
fig, ax = plt.subplots(figsize=(13, 5))
x = list(range(len(comparison_df)))
w = 0.35
ax.bar([i - w/2 for i in x], comparison_df['BB L2'],        width=w, label='BB L2',         color='#1f77b4')
ax.bar([i + w/2 for i in x], comparison_df['WB L3 (best)'], width=w, label='WB L3 (best)', color='#2ca02c')
ax.axhline(mean_bb, color='#1f77b4', linestyle='--', linewidth=1.2,
           label=f'BB L2 global avg ({mean_bb:.2f})')
ax.axhline(mean_wb, color='#2ca02c', linestyle='--', linewidth=1.2,
           label=f'WB L3 global avg ({mean_wb:.2f})')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df.index, rotation=45, ha='right', fontsize=9)
ax.set_title('RQ2 -- BB L2 vs WB L3: Mean Recovered AUC per Source Dataset',
             fontsize=13, pad=12)
ax.set_xlabel('Source (train) dataset', fontsize=11)
ax.set_ylabel('Mean recovered AUC across 14 targets', fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='red', linestyle=':', alpha=0.4, linewidth=0.8)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig('outputs/rq2_bb_vs_wb_bar.png', dpi=300)
plt.show()
print('Saved -> outputs/rq2_bb_vs_wb_generalisation.csv  &  rq2_bb_vs_wb_bar.png')
